# ECCO p02734 Knapsack for all segments
#### Payload type: CPU- and memory-bound O(N*S) DP over Python objects

Source: [ECCO edit/test row 738](https://huggingface.co/datasets/CodeEff/ECCO/viewer/edit/test?row=738) - the slow submission, verbatim, wrapped as `solve(text)`.

In [ ]:
%load_ext jumper_extension
%perfmonitor_fast_setup

In [ ]:
import random
import zlib

In [ ]:
# Inputs scaled within the problem's constraints, fixed by seed
def generate_input(seed, length=1100, target=1100):
    """N = S = 1100 with A_i in [1, S]; the constraints allow N, S up to 3000."""
    rng = random.Random(seed)
    values = " ".join(str(rng.randint(1, target)) for _ in range(length))
    return f"{length} {target}\n{values}\n"


scaled_inputs = [generate_input(seed) for seed in range(3)]

In [ ]:
import sys
def main(text):
    input = iter(text.splitlines(keepends=True)).__next__
    N,S = list(map(int, input().split()))
    A = list(map(int, input().split()))

    dp0 = [[Mint() for _ in range(S+3001)] for _ in range(N+1)]
    dp1 = [[Mint() for _ in range(S+3001)] for _ in range(N+1)]
    dp2 = [[Mint() for _ in range(S+3001)] for _ in range(N+1)]
    dp0[0][0] += 1
    for i in range(N):
        a = A[i]
        dp0i = dp0[i]
        dp1i = dp1[i]
        dp2i = dp2[i]
        dp0n = dp0[i+1]
        dp1n = dp1[i+1]
        dp2n = dp2[i+1]
        for j in range(S+1):
            dp0ij = dp0i[j]
            dp1ij = dp1i[j]
            dp1nja = dp1n[j+a]
            dp2nja = dp2n[j+a]

            dp0n[j] += dp0ij
            dp1nja += dp0ij * (i+1)
            dp2nja += dp0ij * (i+1) * (N-i)

            dp1n[j] += dp1ij
            dp1nja += dp1ij
            dp2nja += dp1ij * (N-i)

            dp2n[j] += dp2i[j]
    return str(dp2[N][S])

MOD = 998244353
class Mint:
    def __init__(self, value=0):
        self.value = value % MOD
        if self.value < 0: self.value += MOD

    @staticmethod
    def get_value(x): return x.value if isinstance(x, Mint) else x

    def inverse(self):
        a, b = self.value, MOD
        u, v = 1, 0
        while b:
            t = a // b
            b, a = a - t * b, b
            v, u = u - t * v, v
        if u < 0: u += MOD
        return u

    def __repr__(self): return str(self.value)
    def __eq__(self, other): return self.value == other.value
    def __neg__(self): return Mint(-self.value)
    def __hash__(self): return hash(self.value)
    def __bool__(self): return self.value != 0

    def __iadd__(self, other):
        self.value += Mint.get_value(other)
        if self.value >= MOD: self.value -= MOD
        return self
    def __add__(self, other):
        new_obj = Mint(self.value)
        new_obj += other
        return new_obj
    __radd__ = __add__

    def __isub__(self, other):
        self.value -= Mint.get_value(other)
        if self.value < 0: self.value += MOD
        return self
    def __sub__(self, other):
        new_obj = Mint(self.value)
        new_obj -= other
        return new_obj
    def __rsub__(self, other):
        new_obj = Mint(Mint.get_value(other))
        new_obj -= self
        return new_obj

    def __imul__(self, other):
        self.value = self.value * Mint.get_value(other) % MOD
        return self
    def __mul__(self, other):
        new_obj = Mint(self.value)
        new_obj *= other
        return new_obj
    __rmul__ = __mul__

    def __ifloordiv__(self, other):
        other = other if isinstance(other, Mint) else Mint(other)
        self *= other.inverse()
        return self
    def __floordiv__(self, other):
        new_obj = Mint(self.value)
        new_obj //= other
        return new_obj
    def __rfloordiv__(self, other):
        new_obj = Mint(Mint.get_value(other))
        new_obj //= self
        return new_obj


answers = [main(text) for text in scaled_inputs]
checksum = sum(zlib.crc32(answer.encode()) for answer in answers)

In [ ]:
%perfmonitor_ai_review --benchmark --replay-mode full